In [23]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel

#| echo: false
#| output: false
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow-cpu==2.21.0 (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tensorflow-cpu==2.21.0


# 📥 Étape 1 : Acquisition des Données & Multi-Sources (Squelette Étudiant)

Cette étape correspond au premier chapitre du pipeline de Data Science. L'objectif est d'identifier, d'importer et de consolider vos jeux de données bruts issus de différentes sources (fichiers CSV locaux, requêtes API, bases de données, etc.).

### 1. Initialisation de l'environnement

In [24]:
#| echo: false
#| output: falseimport os
import os  
import sys
import pandas as pd
import numpy as np

# Ajout du dossier parent au chemin de recherche des modules
sys.path.append(os.path.abspath('..'))

print("Libraries importées avec succès ! Prêt pour l'étape d'Acquisition des Données.")

Libraries importées avec succès ! Prêt pour l'étape d'Acquisition des Données.


### 2. Chargement de la source de données principale

L'acquisition de la donnée constitue le socle de notre démarche. Dans cette cellule, nous avons importé notre jeu de données brut depuis le dépôt officiel Kaggle. Ce dataset massif contient plus de 114 000 morceaux caractérisés par de multiples variables audio (danceability, energy, tempo, etc.), ce qui est idéal pour notre problématique.


In [25]:
#| output: false
from pathlib import Path
import os

# Chercher data/raw/dataset.csv en remontant depuis le dossier courant
_search = Path(os.getcwd()).resolve()
_found  = None
for _ in range(6):
    candidate = _search / 'data' / 'raw' / 'dataset.csv'
    if candidate.exists():
        _found = candidate
        break
    _search = _search.parent

if _found is None:
    raise FileNotFoundError(
        "dataset.csv introuvable. Placez-le dans data/raw/ ou configurez vos credentials Kaggle."
    )

df_main = pd.read_csv(_found, encoding='latin-1')
print(f"Dataset chargé : {df_main.shape}  ({_found})")
df_main.head()

Dataset chargé : (114000, 21)  (C:\Users\jacqu\Desktop\Projets coding\Data science\aptispace-datascience-projet\data\raw\dataset.csv)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


### 3. Intégration de données secondaires (Multi-Sources)

La popularité ne s'expliquant potentiellement pas que par le son pur, nous avons fait le choix d'intégrer des données expertes tierces. Ces données viennent qualifier les 114 genres musicaux (en associant par exemple une énergie moyenne attendue et un coefficient de popularité sectoriel).


In [26]:
# TODO: Récupérer/générer ou charger vos données secondaires
categories_info = pd.DataFrame({
    'track_genre': ['acoustic', 'pop', 'rock', 'hip-hop', 'jazz', 
                    'classical', 'electronic', 'r-n-b', 'latin', 'metal'],
    'description': ['Sons naturels', 'Grand public', 'Guitares électriques',
                    'Rap & Beat', 'Improvisation', 'Musique savante',
                    'Synthétiseurs', 'Rythme & Blues', 'Influences latines',
                    'Son agressif'],
    'energie_moyenne_attendue': [0.4, 0.7, 0.8, 0.65, 0.4,
                                  0.2, 0.85, 0.6, 0.75, 0.9],
    'coef_popularite': [0.9, 1.5, 1.2, 1.4, 0.8,
                        0.7, 1.1, 1.3, 1.2, 0.8]
})

print("Données secondaires prêtes :")
categories_info.head()

Données secondaires prêtes :


,track_genre,description,energie_moyenne_attendue,coef_popularite
0,acoustic,Sons naturels,0.40,0.9
1,pop,Grand public,0.70,1.5
2,rock,Guitares électriques,0.80,1.2
3,hip-hop,Rap & Beat,0.65,1.4
4,jazz,Improvisation,0.40,0.8


### 4. Fusion des sources (Optionnel)

Afin de consolider notre base d'apprentissage, nous avons opéré une jointure gauche (`pd.merge`) entre le dataset Spotify et notre référentiel de genres. Nous obtenons ainsi un Dataframe unifié, prêt pour le nettoyage.


In [27]:
# TODO: Effectuer la jointure de vos tables si nécessaire
df_merged = pd.merge(df_main, categories_info, on='track_genre', how='left')
df_merged.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,description,energie_moyenne_attendue,coef_popularite
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic,Sons naturels,0.4,0.9
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic,Sons naturels,0.4,0.9
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic,Sons naturels,0.4,0.9
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic,Sons naturels,0.4,0.9
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic,Sons naturels,0.4,0.9


### 5. Consignation des données d'entrée brutes

Sauvegardez l'état brut de vos données d'entrée pour la suite du pipeline.

In [28]:
# Sauvegarde du fichier brute fusionné si applicable
print("Acquisition terminée avec succès !")

Acquisition terminée avec succès !
